In [ ]:
!pip install sacrebleu==2.5.1

In [ ]:
import csv
import math
import os
import random
import time
from array import array
from enum import Enum
from pathlib import Path

import numpy as np
import sacrebleu
import sentencepiece as spm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset

In [ ]:
class Languages(Enum):
    ENG = 0
    VIE = 1


SRC_PAD_IDX    = None
TRG_PAD_IDX    = None
SRC_VOCAB_SIZE = None
TRG_VOCAB_SIZE = None
BATCH_SIZE     = 32

# Early Stopping Criteria
PATIENCE       = 2
DELTA          = 0.001

SENTENCE_PIECE_DATA = "/kaggle/input/models/dahyuntw/viet-sentencepiece/other/default/1/viet-sentencepiece"
SAVE_DIR            = "/kaggle/working/models/"
FROM_LANG           = Languages.VIE

ENG_MODEL_PATH = os.path.join(SENTENCE_PIECE_DATA, "english_phomt.model")
VIE_MODEL_PATH = os.path.join(SENTENCE_PIECE_DATA, "vietnamese_phomt.model")

TRAIN_EN_PATH = "/kaggle/input/datasets/dahyuntw/phomt-filtered/PhoMT_filtered/train/train.en"
TRAIN_VI_PATH = "/kaggle/input/datasets/dahyuntw/phomt-filtered/PhoMT_filtered/train/train.vi"
DEV_EN_PATH   = "/kaggle/input/datasets/dahyuntw/phomt-filtered/PhoMT_filtered/dev/dev.en"
DEV_VI_PATH   = "/kaggle/input/datasets/dahyuntw/phomt-filtered/PhoMT_filtered/dev/dev.vi"
TEST_EN_PATH  = "/kaggle/input/datasets/dahyuntw/phomt-filtered/PhoMT_filtered/test/test.en"
TEST_VI_PATH  = "/kaggle/input/datasets/dahyuntw/phomt-filtered/PhoMT_filtered/test/test.vi"

# The scheduler will make learning rate be a much smaller value
# lr * 1 / sqrt(d_model) -> Choose 1.0 (dont scale up or down, we'll use the scheduler)
LEARNING_RATE    = 1.0
SCHEDULER_FACTOR = 1.0
NUM_EPOCHS       = 3
SMOOTHING_FACTOR = 0.1
TRAIN_MODEL_PATH = "/kaggle/input/models/dahyuntw/vie2eng/pytorch/default/3/vie2eng/best_model.pth"
L2_LAMBDA        = 1e-5

# MODEL ARCHITECTURE
HIDDEN_DIM       = 512
NUM_LAYERS       = 6
NUM_HEADS        = 8
MAX_SEQ_LEN      = 64
DROPOUT_RATE     = 0.1

# INFERENCE
class INF_METHOD(Enum):
    GREEDY = 0
    BEAM = 1
    
DECODING_METHOD = INF_METHOD.GREEDY  # "greedy" or "beam"
BEAM_SIZE = 5
RANDOM_SEED = 42


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def is_configured_path(path):
    return path not in (None, "", "PLACEHOLDER")


seed_everything(RANDOM_SEED)

## Data Prep

In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, src_path, tgt_path, src_sp, trg_sp, max_len=64):
        self.src_path = Path(src_path)
        self.tgt_path = Path(tgt_path)
        self.src_sp = src_sp
        self.trg_sp = trg_sp
        self.max_len = max_len
        self.src_offsets = array("Q")
        self.tgt_offsets = array("Q")
        self._src_file = None
        self._tgt_file = None

        if not self.src_path.is_file() or not self.tgt_path.is_file():
            raise FileNotFoundError(
                f"Missing aligned data files: {self.src_path}, {self.tgt_path}"
            )
        self._build_index()

    def _build_index(self):
        with self.src_path.open("rb") as src_file, self.tgt_path.open("rb") as tgt_file:
            line_number = 0
            while True:
                src_offset = src_file.tell()
                tgt_offset = tgt_file.tell()
                src_line = src_file.readline()
                tgt_line = tgt_file.readline()

                if bool(src_line) != bool(tgt_line):
                    raise ValueError(f"Misaligned files after line {line_number}")
                if not src_line:
                    break

                line_number += 1
                if not src_line.strip() or not tgt_line.strip():
                    raise ValueError(f"Empty aligned sentence at line {line_number}")

                self.src_offsets.append(src_offset)
                self.tgt_offsets.append(tgt_offset)

    def _open_files(self):
        if self._src_file is None:
            self._src_file = self.src_path.open("rb")
            self._tgt_file = self.tgt_path.open("rb")

    def __getitem__(self, idx):
        self._open_files()
        self._src_file.seek(self.src_offsets[idx])
        self._tgt_file.seek(self.tgt_offsets[idx])

        src_text = self._src_file.readline().decode("utf-8").strip()
        tgt_text = self._tgt_file.readline().decode("utf-8").strip()
        src_ids = self.src_sp.encode(src_text, out_type=int)
        tgt_ids = self.trg_sp.encode(tgt_text, out_type=int)

        if len(src_ids) > self.max_len:
            raise ValueError(f"Source at index {idx} exceeds {self.max_len} tokens")
        if len(tgt_ids) + 2 > self.max_len:
            raise ValueError(f"Target at index {idx} exceeds {self.max_len} tokens including BOS/EOS")

        labels = [self.trg_sp.bos_id(), *tgt_ids, self.trg_sp.eos_id()]
        return {
            "input_ids": torch.tensor(src_ids, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
            "src_text": src_text,
            "tgt_text": tgt_text,
        }

    def __len__(self):
        return len(self.src_offsets)

    def __getstate__(self):
        state = self.__dict__.copy()
        state["_src_file"] = None
        state["_tgt_file"] = None
        return state

    def __del__(self):
        for handle_name in ("_src_file", "_tgt_file"):
            handle = getattr(self, handle_name, None)
            if handle is not None:
                handle.close()


class TranslationCollator:
    def __init__(self, src_pad_id, tgt_pad_id):
        self.src_pad_id = src_pad_id
        self.tgt_pad_id = tgt_pad_id

    def __call__(self, batch):
        input_ids = pad_sequence(
            [item["input_ids"] for item in batch],
            batch_first=True,
            padding_value=self.src_pad_id,
        )
        labels = pad_sequence(
            [item["labels"] for item in batch],
            batch_first=True,
            padding_value=self.tgt_pad_id,
        )
        return {
            "input_ids": input_ids,
            "labels": labels,
            "src_text": [item["src_text"] for item in batch],
            "tgt_text": [item["tgt_text"] for item in batch],
        }


def load_tokenizer_model(model_path):
    if not Path(model_path).is_file():
        raise FileNotFoundError(f"Tokenizer model not found: {model_path}")
    return spm.SentencePieceProcessor(model_file=str(model_path))


eng_sp = load_tokenizer_model(ENG_MODEL_PATH)
vie_sp = load_tokenizer_model(VIE_MODEL_PATH)

tokenizers = {
    Languages.ENG: (eng_sp, vie_sp),
    Languages.VIE: (vie_sp, eng_sp),
}

split_paths = {
    "train": {
        Languages.ENG: (TRAIN_EN_PATH, TRAIN_VI_PATH),
        Languages.VIE: (TRAIN_VI_PATH, TRAIN_EN_PATH),
    },
    "dev": {
        Languages.ENG: (DEV_EN_PATH, DEV_VI_PATH),
        Languages.VIE: (DEV_VI_PATH, DEV_EN_PATH),
    },
    "test": {
        Languages.ENG: (TEST_EN_PATH, TEST_VI_PATH),
        Languages.VIE: (TEST_VI_PATH, TEST_EN_PATH),
    },
}

src_tkn, trg_tkn = tokenizers[FROM_LANG]
SRC_VOCAB_SIZE = src_tkn.get_piece_size()
TRG_VOCAB_SIZE = trg_tkn.get_piece_size()
SRC_PAD_IDX = src_tkn.pad_id()
TRG_PAD_IDX = trg_tkn.pad_id()


def make_dataset(split):
    src_path, tgt_path = split_paths[split][FROM_LANG]
    return TranslationDataset(
        src_path,
        tgt_path,
        src_tkn,
        trg_tkn,
        max_len=MAX_SEQ_LEN,
    )


train_dataset = make_dataset("train")
val_dataset = make_dataset("dev")
test_dataset = make_dataset("test")

collator = TranslationCollator(SRC_PAD_IDX, TRG_PAD_IDX)
train_generator = torch.Generator().manual_seed(RANDOM_SEED)
pin_memory = torch.cuda.is_available()

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collator,
    generator=train_generator,
    pin_memory=pin_memory,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collator,
    pin_memory=pin_memory,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collator,
    pin_memory=pin_memory,
)

print(f"train={len(train_dataset):,}, dev={len(val_dataset):,}, test={len(test_dataset):,}")

## Helper

In [ ]:
def create_pad_mask(input_ids, pad_idx):
    return (input_ids == pad_idx).unsqueeze(1).unsqueeze(1)


def create_causal_mask(seq_len, device):
    causal_mask = torch.triu(
        torch.ones((seq_len, seq_len), dtype=torch.bool, device=device),
        diagonal=1,
    )
    return causal_mask.unsqueeze(0).unsqueeze(0)


def create_decoder_mask(src, trg, src_pad_idx, trg_pad_idx):
    src_pad_mask = create_pad_mask(src, src_pad_idx)
    trg_pad_mask = create_pad_mask(trg, trg_pad_idx)
    trg_causal_mask = create_causal_mask(trg.size(1), trg.device)
    return src_pad_mask, trg_pad_mask, trg_causal_mask


class Scheduler(torch.optim.lr_scheduler.LambdaLR):
    def __init__(self, optimizer, d_model, warmup_steps=4000, scale_factor=1.0):
        self.d_model = d_model
        self.warmup_steps = warmup_steps
        self.scale_factor = scale_factor
        super().__init__(optimizer, self.lr_lambda)

    def lr_lambda(self, step):
        step = max(step, 1)
        return self.scale_factor * (self.d_model ** -0.5) * min(
            step ** -0.5,
            step * self.warmup_steps ** -1.5,
        )

## Embedder

In [ ]:
class Embedder(nn.Module):
    def __init__(self, vocab_size, d_model):
      """
        vocab_size: the size of the vocabulary
        d_model: the dimension of the embedding vector for EACH token
      """
      super().__init__()
      self.embed = nn.Embedding(vocab_size, d_model)

    def forward(self, x):
      return self.embed(x)

## Positional Encoding

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_len=64):
      super().__init__()

      position = torch.arange(max_seq_len, dtype=torch.float).unsqueeze(1)
      div_term = torch.exp(
          torch.arange(0, d_model, 2).float() * (-math.log(10000) / d_model)
      )

      pe = torch.zeros(max_seq_len, d_model)
      pe[:, 0::2] = torch.sin(position * div_term)
      pe[:, 1::2] = torch.cos(position * div_term)
      pe = pe.unsqueeze(0)

      assert pe.shape == (1, max_seq_len, d_model)
      self.register_buffer("pe", pe, persistent=False)

    def forward(self, x):
      seq_len = x.size(1)
      if seq_len > self.pe.size(1):
          raise ValueError(
              f"Sequence length {seq_len} exceeds positional capacity {self.pe.size(1)}"
          )
      return x + self.pe[:, :seq_len, :]

## Feed-Forward Network

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff=2048, dropout=0.1):
        super().__init__()
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )

    def forward(self, x):
        x = self.ffn(x)
        return x

## Attention

In [ ]:
# nn.MultiheadAttention can be used instead but for learning purposes, I built one from scratch
# https://pytorch.org/docs/stable/generated/torch.nn.MultiheadAttention.html

class MultiHeadAttention(nn.Module):
  def __init__(self, d_model, num_heads, drop_out=0.1):
    """
      d_model: the dimension of the embedding vector for EACH token
      num_heads: the number of attention heads
      drop_out: the dropout rate
    """
    super().__init__()
    assert d_model % num_heads == 0, "MultiHeadAttention.__init__(): d_model must be divisible by num_heads"
    self.d_model   = d_model
    self.num_heads = num_heads
    self.head_dim  = d_model // num_heads

    self.q_linear = nn.Linear(d_model, d_model)
    self.v_linear = nn.Linear(d_model, d_model)
    self.k_linear = nn.Linear(d_model, d_model)

    # Final linear projection after concatenating heads (Learns how to merge all heads back into a single representation)
    self.out_proj = nn.Linear(d_model, d_model)

    self.dropout = nn.Dropout(drop_out)

  def forward(self, q, k, v, mask=None, cache=None):
    '''
     `cache` should be a dict of 
     {
      "K": Tensor [B, num_heads, L_cached, d_head],
      "V": Tensor [B, num_heads, L_cached, d_head],
     }
    '''
    # x: [batch_size, seq_len, d_model] or [B, L, D]
    B, L, D = q.shape

    # Linear projections (trainable parameters) [B, L, D]
    Q = self.q_linear(q)
    K = self.k_linear(k)
    V = self.v_linear(v)

    # Split into heads: [B, L, num_heads, head_dim] → transpose → [B, num_heads, L, head_dim]
    Q = Q.view(B, -1, self.num_heads, self.head_dim).transpose(1, 2)
    K = K.view(B, -1, self.num_heads, self.head_dim).transpose(1, 2)
    V = V.view(B, -1, self.num_heads, self.head_dim).transpose(1, 2)

    if cache is not None:
        K = torch.cat([cache['K'], K], dim=2)
        V = torch.cat([cache['V'], V], dim=2)
    new_cache = {'K': K, 'V': V}
    
    # Scaled dot-product attention
    # Q      = [B, num_heads, L,        head_dim]
    # K_T    = [B, num_heads, head_dim, L]
    # scores = [B, num_heads, L,        L]
    scale  = 1 / math.sqrt(self.head_dim) # prevents the softmax function from saturation -> extremely small gradients
    scores = torch.matmul(Q, K.transpose(-2, -1)) * scale

    # Preventing the decoder from attending to future tokens (causal mask)
    # Ignoring <pad> tokens
    if mask is not None:
      scores = scores.masked_fill(mask, float('-inf')) # masked_fill(mask, value) applies 'value' where 'mask == True'

    attn = torch.softmax(scores, dim=-1) # [B, num_heads, L, L]
    attn = self.dropout(attn)

    output = torch.matmul(attn, V) # [B, num_heads, L, head_dim]

    # Concatenate heads
    output = output.transpose(1, 2).contiguous().view(B, L, D)  # [B, L, D]
    output = self.out_proj(output)
    assert output.shape == (B, L, D)

    return output, new_cache

## Encoder

In [ ]:
class EncoderLayer(nn.Module):
  def __init__(self, d_model, num_heads, ffn_dim, drop_out):
    super().__init__()
    # self.attn = MultiHeadAttention(d_model, num_heads, is_enc=True) # torch version
    self.attn = MultiHeadAttention(d_model, num_heads) # Minh's
    self.ffn  = FeedForward(d_model, ffn_dim, drop_out)

    # Normalization: https://medium.com/@florian_algo/batchnorm-and-layernorm-2637f46a998b
    # Use normalization to prevent gradient vanishing 
    # -> ensuring that the feature values fall within the range where the activation function 
    #    is more sensitive to inputs, thereby avoiding gradient vanishing and speeding up convergence.
    self.norm1 = nn.LayerNorm(d_model)
    self.norm2 = nn.LayerNorm(d_model)
    self.dropout1 = nn.Dropout(drop_out)
    self.dropout2 = nn.Dropout(drop_out)

  def forward(self, x, mask=None):
    # 1. Attention
    x2 = self.norm1(x)
    attn1_output, _ = self.attn(x2, x2, x2, mask) # minh's multi-head attn
    x = x + self.dropout1(attn1_output) 
    # x = x + self.dropout1(self.attn(x2, x2, x2, padding_mask=mask)) # torch

    # 2. Feed-forward
    x2 = self.norm2(x)
    x = x + self.dropout2(self.ffn(x2))
    return x

In [ ]:
class TransformerEncoder(nn.Module):
  def __init__(self,
               vocab_size,
               pad_idx,
               d_model=512,
               num_layers=6,
               num_attn_heads=8,
               ffn_dim=2048,
               max_seq_len=64,
               dropout=0.1):
    super().__init__()
    self.d_model = d_model
    self.pad_idx = pad_idx

    self.embedder = Embedder(vocab_size, d_model)
    self.positional_encoding = PositionalEncoding(d_model, max_seq_len)
    self.encoder_layers = nn.ModuleList([
        EncoderLayer(d_model, num_attn_heads, ffn_dim, dropout)
        for _ in range(num_layers)
    ])
    self.dropout = nn.Dropout(dropout)
    self.norm = nn.LayerNorm(d_model)

  def forward(self, input_ids):
    batch_size, seq_len = input_ids.shape
    src_mask = create_pad_mask(input_ids, self.pad_idx)

    x = self.embedder(input_ids) * math.sqrt(self.d_model)
    assert x.shape == (batch_size, seq_len, self.d_model)

    x = self.positional_encoding(x)
    x = self.dropout(x)

    for encoder_layer in self.encoder_layers:
      x = encoder_layer(x, src_mask)

    return self.norm(x)

## Decoder

In [ ]:
class DecoderLayer(nn.Module):
  def __init__(self, d_model, num_heads, ffn_dim, drop_out):
    super().__init__()
    self.attn1 = MultiHeadAttention(d_model, num_heads) # Minh's
    self.attn2 = MultiHeadAttention(d_model, num_heads) # Minh's

    self.ffn  = FeedForward(d_model, ffn_dim, drop_out)

    self.norm1 = nn.LayerNorm(d_model)
    self.norm2 = nn.LayerNorm(d_model)
    self.norm3 = nn.LayerNorm(d_model)

    self.dropout1 = nn.Dropout(drop_out)
    self.dropout2 = nn.Dropout(drop_out)
    self.dropout3 = nn.Dropout(drop_out)

  def forward(self, x, enc_output, dec_masks, cache):
    """
    x: [batch size, target length, hidden dim]
    enc_output: [batch size, source length, hidden dim]
    dec_mask: (enc_dec_mask [B, 1, 1, L], combined_mask [1, 1, L, L])
    Note:
    - In "encoder-decoder attention" layers, the queries come from the previous decoder layer,
      and the memory keys and values come from the output of the encoder (https://arxiv.org/abs/1706.03762)
    """
    enc_dec_mask    = dec_masks[0]
    dec_pad_mask    = dec_masks[1]
    dec_causal_mask = dec_masks[2]

    # 1. Causally-masked Attention
    self_attn_cache = cache if cache != None else None
    x2 = self.norm1(x)
    attn1_output, attn1_cache = self.attn1(x2, x2, x2, dec_pad_mask | dec_causal_mask, self_attn_cache) # Only cache causal attention
    x = x + self.dropout1(attn1_output) # Minh's

    # 2. Encoder-Decoder Attention
    x2 = self.norm2(x)
    attn2_output, _ = self.attn2(x2, enc_output, enc_output, enc_dec_mask) # enc_output = k = v -> same for every decoding step -> no caching
    x = x + self.dropout2(attn2_output) # Minh's

    # 3. Feed-forward
    x2 = self.norm3(x)
    x = x + self.dropout3(self.ffn(x2))

    return x, attn1_cache

In [ ]:
class TransformerDecoder(nn.Module):
  def __init__(self,
               vocab_size,
               src_pad_idx,
               trg_pad_idx,
               d_model=512,
               num_layers=6,
               num_attn_heads=8,
               ffn_dim=2048,
               max_seq_len=64,
               dropout=0.1):
    super().__init__()

    self.d_model = d_model
    self.vocab_size = vocab_size
    self.src_pad_idx = src_pad_idx
    self.trg_pad_idx = trg_pad_idx

    self.embedder = Embedder(vocab_size, d_model)
    self.positional_encoding = PositionalEncoding(d_model, max_seq_len)
    self.decoder_layers = nn.ModuleList([
        DecoderLayer(d_model, num_attn_heads, ffn_dim, dropout)
        for _ in range(num_layers)
    ])

    self.dropout = nn.Dropout(dropout)
    self.norm = nn.LayerNorm(d_model)

  def forward(self, src_ids, target_ids, encoder_outputs, cache=None, last_token_only=False):
    batch_size, seq_len = target_ids.shape
    dec_masks = create_decoder_mask(
        src_ids,
        target_ids,
        self.src_pad_idx,
        self.trg_pad_idx,
    )

    x = self.embedder(target_ids) * math.sqrt(self.d_model)
    assert x.shape == (batch_size, seq_len, self.d_model)

    x = self.positional_encoding(x)
    x = self.dropout(x)

    new_caches = []
    for layer_idx, decoder_layer in enumerate(self.decoder_layers):
      layer_cache = None if cache is None else cache[layer_idx]
      x, new_layer_cache = decoder_layer(x, encoder_outputs, dec_masks, layer_cache)
      new_caches.append(new_layer_cache)

    # Generation only needs vocabulary scores for the newest position.
    output = self.norm(x[:, -1:, :] if last_token_only else x)
    output = torch.matmul(output, self.embedder.embed.weight.transpose(0, 1))
    output_len = 1 if last_token_only else seq_len
    assert output.shape == (batch_size, output_len, self.vocab_size)

    return output, new_caches

## Transformer

In [ ]:
class Transformer(nn.Module):
  def __init__(self,
               src_vocab_size,
               trg_vocab_size,
               src_pad_idx,
               trg_pad_idx,
               d_model=512,
               max_seq_len=64,
               num_layers=6,
               num_attn_heads=8,
               dropout=0.1):
    super().__init__()
    self.src_vocab_size = src_vocab_size
    self.trg_vocab_size = trg_vocab_size
    self.encoder = TransformerEncoder(
        src_vocab_size,
        src_pad_idx,
        d_model=d_model,
        num_layers=num_layers,
        num_attn_heads=num_attn_heads,
        max_seq_len=max_seq_len,
        dropout=dropout,
    )
    self.decoder = TransformerDecoder(
        trg_vocab_size,
        src_pad_idx,
        trg_pad_idx,
        d_model=d_model,
        num_layers=num_layers,
        num_attn_heads=num_attn_heads,
        max_seq_len=max_seq_len,
        dropout=dropout,
    )

  def encode(self, src):
      return self.encoder(src)

  def decode(self, src, trg, enc_output, cache=None, last_token_only=False):
      return self.decoder(src, trg, enc_output, cache, last_token_only=last_token_only)

  def forward(self, src, trg):
    assert src.device == trg.device
    enc_output = self.encode(src)
    dec_output, _ = self.decode(src, trg, enc_output)
    return dec_output


def build_model():
    return Transformer(
        src_vocab_size=SRC_VOCAB_SIZE,
        trg_vocab_size=TRG_VOCAB_SIZE,
        src_pad_idx=SRC_PAD_IDX,
        trg_pad_idx=TRG_PAD_IDX,
        d_model=HIDDEN_DIM,
        max_seq_len=MAX_SEQ_LEN,
        num_layers=NUM_LAYERS,
        num_attn_heads=NUM_HEADS,
        dropout=DROPOUT_RATE,
    )


def checkpoint_config():
    return {
        "hidden_dim": HIDDEN_DIM,
        "num_layers": NUM_LAYERS,
        "num_heads": NUM_HEADS,
        "max_seq_len": MAX_SEQ_LEN,
        "dropout_rate": DROPOUT_RATE,
        "src_vocab_size": SRC_VOCAB_SIZE,
        "trg_vocab_size": TRG_VOCAB_SIZE,
        "from_lang": FROM_LANG.name,
    }

## Metrics

In [ ]:
def accuracy(output, target, topk=(1,)):
    """Computes the precision@k for the specified values of k"""
    maxk = max(topk)
    batch_seq_len = target.size(0)

    _, pred = output.topk(maxk, 1, True, True)
    pred = pred.t()
    correct = pred.eq(target.view(1, -1).expand_as(pred))

    res = []
    for k in topk:
        correct_k = correct[:k].view(-1).float().sum(0)
        res.append(correct_k.mul_(100.0 / batch_seq_len))
    return res

class AverageMeter(object):
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

def l2_regularization(model, l2_lambda=1.0):
    l2_norm = 0.0
    for name, param in model.named_parameters():
        if 'bias' not in name and 'norm' not in name:  # skip biases and LayerNorms
            l2_norm += torch.sum(param.pow(2))
    return l2_lambda * l2_norm

In [ ]:
def log_to_csv(file_name, epoch, train_loss, val_loss, learning_rate):
    with open(file_name, "a", encoding="utf-8", newline="") as log_file:
        writer = csv.writer(log_file)
        writer.writerow([epoch, train_loss, val_loss, learning_rate])

## Translate Algo

In [ ]:
def greedy_batch(
    model,
    src,
    trg_tokenizer,
    max_seq_len=MAX_SEQ_LEN
):
    trg_output = []
    batch_len, _ = src.shape
    for sentence_idx in range(batch_len):
        src_sentence = src[sentence_idx, :].unsqueeze(0)
        out_sentence = greedy_search(model, src_sentence, max_seq_len, trg_tokenizer)
        trg_output.append(out_sentence)
        
    return trg_output
    
def greedy_search(
    model,
    src,
    max_seq_len,
    trg_tokenizer
) -> str:
    # 1. Create a translation tensor, init the first token to <bos>/<sos> (beginning/start of seq)
    y_trg = torch.tensor([[trg_tokenizer.bos_id()]], device=src.device) # Current size [1, 1], will be [1, max_seq_len] eventually
    enc_output = model.encode(src) # [batch size, source length, hidden dim]

    last_cache = None
    for _ in range(max_seq_len - 1):
        dec_output, _ = model.decode(src, y_trg, enc_output) # [1, trg_seq_len, trg_vocab_size]
        # dec_output, last_cache = model.decode(src, y_trg[:, -1:], enc_output, last_cache) # [1, trg_seq_len, trg_vocab_size]
        
        # Get the last predicted token (at position trg_seq_len - 1)
        # Return the token that has the highest vocab score
        next_token = dec_output[:, -1, :].argmax(-1) # Return the index of the token with highest value
        y_trg = torch.cat((y_trg, next_token.unsqueeze(0)), dim=-1)

        if next_token.item() == trg_tokenizer.eos_id(): # Early break if <eos> has been generated
            break
    return trg_tokenizer.decode(y_trg.squeeze().tolist())
    
def beam_batch(
    model,
    src,
    trg_tokenizer,
    beam_size=3,
    max_seq_len=MAX_SEQ_LEN,
    length_penalty=0.75
):
    trg_output = []
    batch_len, _ = src.shape
    for sentence_idx in range(batch_len):
        src_sentence = src[sentence_idx, :].unsqueeze(0)
        out_sentence = beam_search(model, src_sentence, trg_tokenizer, beam_size, max_seq_len, length_penalty)
        trg_output.append(out_sentence)
        
    return trg_output
    
@torch.no_grad()
def beam_search(
    model,
    src,
    trg_tokenizer,
    beam_size=5,
    max_len=64,
    length_penalty=0.75,
) -> str:
    """
    Translate one source sentence using beam search.
    https://medium.com/@jessica_lopez/understanding-greedy-search-and-beam-search-98c1e3cd821d
    
    src: Tensor shaped [1, source_length].
    max_len: Maximum generated tokens, excluding BOS but including EOS.
    length_penalty: 0 disables length normalization.
    """
    if src.ndim != 2 or src.size(0) != 1:
        raise ValueError("src must have shape [1, source_length].")

    if beam_size < 1 or max_len < 1:
        raise ValueError("beam_size and max_len must be positive.")

    bos = trg_tokenizer.bos_id()
    eos = trg_tokenizer.eos_id()
    pad = trg_tokenizer.pad_id()

    if bos < 0 or eos < 0:
        raise ValueError("The tokenizer must define BOS and EOS token IDs.")

    def score(beam):
        # Ref: https://arxiv.org/abs/1609.08144 (7. Decoder - eq. 14)
        tokens, log_prob, _ = beam
        generated_length = tokens.size(1) - 1  # Exclude BOS.
        normalization = ((5.0 + generated_length) / 6.0) ** length_penalty
        return log_prob / normalization

    encoder_output = model.encode(src)

    # Track completion in Python to avoid a GPU read for every beam.
    beams = [
        (
            torch.tensor([[bos]], dtype=torch.long, device=src.device),
            0.0,
            bos == eos,
        )
    ]

    for _ in range(max_len):
        active_tokens = [tokens for tokens, _, finished in beams if not finished]
        if not active_tokens:
            break

        # Unfinished beams share a prefix length, so decode them in one batch.
        prefixes = torch.cat(active_tokens, dim=0)
        active_count = prefixes.size(0)
        output, _ = model.decode(
            src.expand(active_count, -1),
            prefixes,
            encoder_output.expand(active_count, -1, -1),
            last_token_only=True,
        )
        next_log_probs = torch.log_softmax(output[:, -1, :], dim=-1)

        for token_id in {bos, pad}:
            if 0 <= token_id < next_log_probs.size(-1) and token_id != eos:
                next_log_probs[:, token_id] = float("-inf")

        k = min(beam_size, next_log_probs.size(-1))
        top_log_probs, top_token_ids = torch.topk(next_log_probs, k, dim=-1)
        expansions = iter(zip(top_log_probs.tolist(), top_token_ids.tolist()))
        candidates = []

        # Preserve the original candidate order, including completed beams.
        for tokens, cumulative_log_prob, finished in beams:
            if finished:
                candidates.append((tokens, cumulative_log_prob, True))
                continue

            log_probs, token_ids = next(expansions)
            for token_log_prob, token_id in zip(log_probs, token_ids):
                if not math.isfinite(token_log_prob):
                    continue

                next_token = tokens.new_tensor([[token_id]])
                candidates.append(
                    (
                        torch.cat((tokens, next_token), dim=1),
                        cumulative_log_prob + token_log_prob,
                        token_id == eos,
                    )
                )

        if not candidates:
            raise RuntimeError("No valid beam-search candidates remain.")

        beams = sorted(candidates, key=score, reverse=True)[:beam_size]
        if all(finished for _, _, finished in beams):
            break

    best_tokens, _, _ = max(beams, key=score)
    token_ids = best_tokens[0, 1:].tolist()  # Remove BOS.

    if token_ids and token_ids[-1] == eos:
        token_ids.pop()

    return trg_tokenizer.decode(token_ids)

## Train

In [ ]:
TRAIN_PRINT_FREQ = 200


def train(model, optimizer, scheduler, train_loader, criterion, device, epoch_id):
  model.train()
  batch_time_meter = AverageMeter()
  loss_meter = AverageMeter()

  for i, batch in enumerate(train_loader):
    start_time = time.time()

    src = batch["input_ids"].to(device, non_blocking=True)
    trg = batch["labels"].to(device, non_blocking=True)
    trg_input = trg[:, :-1]
    trg_target = trg[:, 1:]

    output = model(src, trg_input)
    preds = output.reshape(-1, output.size(-1))
    flat_target = trg_target.reshape(-1)

    optimizer.zero_grad()
    loss = criterion(preds, flat_target)
    loss.backward()
    optimizer.step()
    scheduler.step()

    valid_tokens = (flat_target != TRG_PAD_IDX).sum().item()
    loss_meter.update(loss.item(), valid_tokens)
    batch_time_meter.update(time.time() - start_time)

    if i % TRAIN_PRINT_FREQ == 0:
      print(
          "Epoch: [{0}][{1}/{2}]\t"
          "Time(s) {time.val:.3f} ({time.avg:.3f})\t"
          "Loss {loss.val:.4f} ({loss.avg:.4f})".format(
              epoch_id,
              i,
              len(train_loader),
              time=batch_time_meter,
              loss=loss_meter,
          )
      )
      batch_time_meter.reset()

  return loss_meter.avg

In [ ]:
VAL_PRINT_FREQ = 100


@torch.inference_mode()
def evaluate_loss(model, data_loader, criterion, device, split_name="Validation"):
    model.eval()
    loss_meter = AverageMeter()

    for i, batch in enumerate(data_loader):
        src = batch["input_ids"].to(device, non_blocking=True)
        trg = batch["labels"].to(device, non_blocking=True)
        trg_input = trg[:, :-1]
        trg_target = trg[:, 1:]

        output = model(src, trg_input)
        preds = output.reshape(-1, output.size(-1))
        flat_target = trg_target.reshape(-1)
        loss = criterion(preds, flat_target)

        valid_tokens = (flat_target != TRG_PAD_IDX).sum().item()
        loss_meter.update(loss.item(), valid_tokens)

        if i % VAL_PRINT_FREQ == 0:
            print(f"{split_name}: [{i}/{len(data_loader)}] Loss {loss_meter.avg:.4f}")

    print(f"* {split_name} average loss: {loss_meter.avg:.4f}")
    return loss_meter.avg


@torch.inference_mode()
def evaluate_translation(model, data_loader, criterion, device, trg_tokenizer):
    """Export sentence chrF++ and token lengths; len_ratio is hyp_tokens / ref_tokens."""
    if DECODING_METHOD not in (INF_METHOD.GREEDY, INF_METHOD.BEAM):
        raise ValueError("DECODING_METHOD must be INF_METHOD.GREEDY or INF_METHOD.BEAM")

    model.eval()
    loss_meter = AverageMeter()
    predictions = []
    references = []
    sources = []
    source_token_counts = []

    for i, batch in enumerate(data_loader):
        src = batch["input_ids"].to(device, non_blocking=True)
        trg = batch["labels"].to(device, non_blocking=True)
        trg_input = trg[:, :-1]
        trg_target = trg[:, 1:]

        output = model(src, trg_input)
        preds = output.reshape(-1, output.size(-1))
        flat_target = trg_target.reshape(-1)
        loss = criterion(preds, flat_target)

        valid_tokens = (flat_target != TRG_PAD_IDX).sum().item()
        loss_meter.update(loss.item(), valid_tokens)

        if DECODING_METHOD == INF_METHOD.GREEDY:
            predictions.extend(greedy_batch(model, src, trg_tokenizer))
        else:
            predictions.extend(beam_batch(model, src, trg_tokenizer, beam_size=BEAM_SIZE))
        references.extend(batch["tgt_text"])
        sources.extend(batch["src_text"])
        source_token_counts.extend((src != SRC_PAD_IDX).sum(dim=1).tolist())

        if i % VAL_PRINT_FREQ == 0:
            print(f"Test: [{i}/{len(data_loader)}] Loss {loss_meter.avg:.4f}")

    if not predictions:
        raise ValueError("Cannot evaluate an empty translation dataset")
    if not (len(sources) == len(references) == len(predictions) == len(source_token_counts)):
        raise ValueError("Source, reference, prediction and token counts must match")

    bleu_metric = sacrebleu.metrics.BLEU(tokenize="13a")
    chrf_metric = sacrebleu.metrics.CHRF(word_order=2)
    bleu = bleu_metric.corpus_score(predictions, [references])
    chrf = chrf_metric.corpus_score(predictions, [references])

    log_dir = os.path.join(SAVE_DIR, f"from_{FROM_LANG.name}")
    os.makedirs(log_dir, exist_ok=True)
    search_name = DECODING_METHOD.name.lower()
    if DECODING_METHOD == INF_METHOD.BEAM:
        search_name += str(BEAM_SIZE)
    csv_path = os.path.join(log_dir, f"test_predictions_{search_name}.csv")
    with open(csv_path, "w", encoding="utf-8-sig", newline="") as csv_file:
        writer = csv.writer(csv_file)
        writer.writerow([
            "source", "ref", "output", "chrf++", "src_chars", "src_tokens",
            "ref_tokens", "hyp_tokens", "len_ratio",
        ])
        for source, reference, hypothesis, src_tokens in zip(
            sources, references, predictions, source_token_counts
        ):
            ref_tokens = len(trg_tokenizer.encode(reference, out_type=int))
            hyp_tokens = len(trg_tokenizer.encode(hypothesis, out_type=int))
            writer.writerow([
                source, reference, hypothesis,
                float(chrf_metric.sentence_score(hypothesis, [reference]).score),
                len(source), src_tokens, ref_tokens, hyp_tokens,
                hyp_tokens / ref_tokens if ref_tokens else None,
            ])
    print(f"Test sentence predictions saved to: {csv_path}")

    metrics = {
        "bleu": float(bleu.score),
        "chrf2": float(chrf.score),
        "bleu_signature": str(bleu_metric.get_signature()),
    }
    print(f"* Test average loss: {loss_meter.avg:.4f}")
    print(f"* Test BLEU: {metrics['bleu']:.4f}")
    print(f"* Test chrF++: {metrics['chrf2']:.4f}")
    print(f"* SacreBLEU signature: {metrics['bleu_signature']}")
    return loss_meter.avg, metrics


def save_test_log(save_dir, avg_loss, metrics):
    log_dir = os.path.join(save_dir, f"from_{FROM_LANG.name}")
    os.makedirs(log_dir, exist_ok=True)
    log_path = os.path.join(log_dir, "test_log.csv")
    exists = os.path.exists(log_path) and os.path.getsize(log_path) > 0

    header = [
        "avg_loss", "corpus_bleu", "chrf2", "bleu_signature",
    ]
    if exists:
        with open(log_path, "r", encoding="utf-8", newline="") as log_file:
            if next(csv.reader(log_file), None) != header:
                raise ValueError(
                    "Existing test_log.csv has a different header. Rename it or "
                    "use a new SAVE_DIR before logging test results."
                )

    with open(log_path, "a", encoding="utf-8", newline="") as log_file:
        writer = csv.writer(log_file)
        if not exists:
            writer.writerow(header)
        writer.writerow([
            avg_loss,
            metrics["bleu"],
            metrics["chrf2"],
            metrics["bleu_signature"],
        ])

    print(f"Test log saved to: {log_path}")
    return log_path

## Main (Training Loop)

In [ ]:
os.makedirs(SAVE_DIR, exist_ok=True)
LOG_FILE_PATH = os.path.join(SAVE_DIR, "log.csv")
BEST_CHECKPOINT_PATH = os.path.join(SAVE_DIR, "best_model.pth")

if not os.path.exists(LOG_FILE_PATH):
    with open(LOG_FILE_PATH, "w", encoding="utf-8", newline="") as log_file:
        csv.writer(log_file).writerow([
            "epoch",
            "train_loss",
            "val_loss",
            "learning_rate",
        ])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = build_model().to(device)
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    betas=(0.9, 0.98),
    eps=1e-9,
)
scheduler = Scheduler(
    optimizer,
    d_model=HIDDEN_DIM,
    scale_factor=SCHEDULER_FACTOR,
)
criterion = nn.CrossEntropyLoss(
    ignore_index=TRG_PAD_IDX,
    label_smoothing=SMOOTHING_FACTOR,
).to(device)


def save_checkpoint(path, epoch, best_val_loss, patience_count):
    state = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "best_val_loss": best_val_loss,
        "patience_count": patience_count,
        "config": checkpoint_config(),
        "torch_rng_state": torch.get_rng_state(),
        "train_generator_state": train_generator.get_state(),
    }
    if torch.cuda.is_available():
        state["cuda_rng_state_all"] = torch.cuda.get_rng_state_all()
    torch.save(state, path)


start_epoch = 0
best_val_loss = float("inf")
patience_count = 0

if is_configured_path(TRAIN_MODEL_PATH):
    checkpoint = torch.load(TRAIN_MODEL_PATH, map_location=device)
    if checkpoint.get("config") not in (None, checkpoint_config()):
        raise ValueError("Checkpoint configuration does not match this notebook")

    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    if "scheduler_state_dict" in checkpoint:
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

    start_epoch = checkpoint.get("epoch", 0)
    best_val_loss = checkpoint.get("best_val_loss", float("inf"))
    patience_count = checkpoint.get("patience_count", 0)

    if "torch_rng_state" in checkpoint:
        torch.set_rng_state(checkpoint["torch_rng_state"].cpu())
    if torch.cuda.is_available() and "cuda_rng_state_all" in checkpoint:
        torch.cuda.set_rng_state_all([state.cpu() for state in checkpoint["cuda_rng_state_all"]])
    if "train_generator_state" in checkpoint:
        train_generator.set_state(checkpoint["train_generator_state"].cpu())

for epoch_id in range(start_epoch, NUM_EPOCHS):
    train_loss = train(
        model,
        optimizer,
        scheduler,
        train_loader,
        criterion,
        device,
        epoch_id,
    )
    val_loss = evaluate_loss(
        model,
        val_loader,
        criterion,
        device,
        split_name="Validation",
    )
    learning_rate = scheduler.get_last_lr()[0]
    log_to_csv(
        LOG_FILE_PATH,
        epoch_id + 1,
        train_loss,
        val_loss,
        learning_rate,
    )
    print(
        f"*** EPOCH {epoch_id + 1}: "
        f"Train Loss {train_loss:.4f}; Val Loss {val_loss:.4f}"
    )

    if val_loss < best_val_loss - DELTA:
        best_val_loss = val_loss
        patience_count = 0
        save_checkpoint(
            BEST_CHECKPOINT_PATH,
            epoch_id + 1,
            best_val_loss,
            patience_count,
        )
        print(f"Saved best checkpoint with validation loss {best_val_loss:.4f}")
    else:
        patience_count += 1

    if (epoch_id + 1) % 10 == 0:
        periodic_path = os.path.join(SAVE_DIR, f"epoch_{epoch_id + 1}.pth")
        save_checkpoint(
            periodic_path,
            epoch_id + 1,
            best_val_loss,
            patience_count,
        )
        print(f"Saved {periodic_path}")

    if patience_count >= PATIENCE:
        print(f"Early stopping after epoch {epoch_id + 1}")
        break

if os.path.isfile(BEST_CHECKPOINT_PATH):
    best_checkpoint = torch.load(BEST_CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(best_checkpoint["model_state_dict"])
    print(
        f"Restored best checkpoint from epoch {best_checkpoint['epoch']} "
        f"with validation loss {best_checkpoint['best_val_loss']:.4f}"
    )

## Testing

In [ ]:
TEST_MODEL_PATH = "/kaggle/input/models/dahyuntw/vie2eng/pytorch/default/2/vie2eng/best_model.pth"

In [ ]:
# if not is_configured_path(TEST_MODEL_PATH):
#     raise ValueError("Set TEST_MODEL_PATH before running the test cell")

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model = build_model().to(device)
# criterion = nn.CrossEntropyLoss(
#     ignore_index=TRG_PAD_IDX,
#     label_smoothing=SMOOTHING_FACTOR,
# ).to(device)

# checkpoint = torch.load(TEST_MODEL_PATH, map_location=device)
# if checkpoint.get("config") not in (None, checkpoint_config()):
#     print(checkpoint.get("config"))
#     print(checkpoint_config())
#     raise ValueError("Checkpoint configuration does not match this notebook")
# model.load_state_dict(checkpoint["model_state_dict"])

# test_loss, test_metrics = evaluate_translation(
#     model,
#     test_loader,
#     criterion,
#     device,
#     trg_tkn,
# )
# LOG_DIR = "/kaggle/working/test_run/"
# save_test_log(LOG_DIR, test_loss, test_metrics)

## Translate

In [ ]:
# if "model" not in globals():
#     raise RuntimeError("Load a trained model before translating")

# vie_sentence = "Ngày mai tôi sẽ về nhà."
# src_ids = src_tkn.encode(vie_sentence, out_type=int)
# if len(src_ids) > MAX_SEQ_LEN:
#     raise ValueError(f"Input exceeds {MAX_SEQ_LEN} tokens")

# src = torch.tensor(src_ids, dtype=torch.long, device=device).unsqueeze(0)
# model.eval()
# translation = beam_search(
#     model,
#     src,
#     trg_tkn,
#     max_len=MAX_SEQ_LEN,
# )
# print(translation)